### Compare TFIDF output from networkx with neo4j

In [21]:
import ast
import logging
import os
import pickle
import random

import pandas as pd
import pyvis
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()

True

### Networkx visualisation using pyvis

In [22]:
EMBEDDING_MODEL = "tfidf_cosine"

INPUT_GRAPH_PATH = os.path.join(
    "..",
    "artifacts",
    "outputs",
    EMBEDDING_MODEL + "_graph",
)

INPUT_CSV_PATH = os.path.join(
    "..",
    "artifacts",
    "outputs",
    f"{EMBEDDING_MODEL}_graph_louvain_cluster.csv",
)

graph = pickle.load(open(INPUT_GRAPH_PATH, "rb"))
pred_df = pd.read_csv(INPUT_CSV_PATH)
THRESHOLD = graph.graph["threshold"]  # To be used by neo4j as the THRESHOLD similarity

In [24]:
# Add page title attribute to nodes

cluster_counts = pred_df["cluster_label"].value_counts()

# Filter cluster_labels with count greater than 1
valid_clusters = cluster_counts[cluster_counts > 1].index

# Update cluster_label column
pred_df.loc[~pred_df["cluster_label"].isin(valid_clusters), "cluster_label"] = pd.NA

random.seed(123)
node_cluster = [id for id in pred_df["cluster_label"].unique() if str(id) != "nan"]

cluster_colors = {}
for cluster_id in set(node_cluster):
    color = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))
    hex_color = "#%02x%02x%02x" % color
    cluster_colors[str(cluster_id)] = hex_color

for node, data in graph.nodes(data=True):
    node_title = data["Page Title"]
    node_ground_truth = data["Combine Group ID"]

    # Find the corresponding row in pred_df
    matching_row = pred_df[pred_df["Page Title"] == node_title]

    # If a matching row is found, update the node's attribute
    if not matching_row.empty:
        predicted_cluster = matching_row["cluster_label"].values[0]
        data["color"] = cluster_colors.get(str(predicted_cluster), "gray")
        data["Predicted Cluster"] = predicted_cluster
        hover_text = f"Page Title: {node_title}\nGround Truth: {str(node_ground_truth)}\nPredicted Cluster: {str(predicted_cluster)}"
        data["title"] = hover_text

g = pyvis.network.Network(notebook=True, select_menu=True, filter_menu=True)
g.from_nx(graph)
g.show("networkx_network_tfidf_1.html")

networkx_network_tfidf_1.html


### Connect to neo4j

In [25]:
uri = "neo4j://localhost:7687"
username = os.getenv("neo4j_username")
password = os.getenv("neo4j_password")
driver = GraphDatabase.driver(uri, auth=(username, password))

In [26]:
NEO4J = {
    "uri": uri,
    "auth": (username, password),
    "database": "tfidf",  # create this database in neo4j first
}

In [27]:
with open("articles.pkl", "rb") as f:
    articles = pickle.load(f)

articles["vector"] = articles["vector"].apply(ast.literal_eval)

documents = articles.to_dict(orient="records")
documents[0].keys()

dict_keys(['id', 'title', 'content', 'meta_description', 'vector', 'ground_truth_label'])

In [28]:
logging.basicConfig(level=logging.INFO)


def clear_db(tx):
    logging.info("Clearing database")
    tx.run("MATCH (n) DETACH DELETE n")


def create_graph_nodes(tx, doc):
    # logging.info("Create nodes")
    tx.run(
        """
    CREATE (d:Article {
        id: $id,
        title: $title, 
        content: $content, 
        meta_desc: $meta_description,
        vector: $vector,
        ground_truth: $ground_truth
    })""",
        id=doc["id"],
        title=doc["title"],
        content=doc["content"],
        meta_description=doc["meta_description"],
        vector=doc["vector"],
        ground_truth=doc["ground_truth_label"],
    )


def create_sim_edges(tx, threshold):
    logging.info("Create edges")
    result = tx.run(
        """
    MATCH (a:Article), (b:Article)
    WHERE a.id < b.id
    WITH a, b, gds.similarity.cosine(a.vector, b.vector) AS similarity
    WHERE similarity > $threshold
    CREATE (a)-[:SIMILAR {similarity: similarity}]->(b)
    """,
        threshold=threshold,
    )


def create_graph_proj(tx):
    # logging.info("Create projection")
    tx.run(
        """
           CALL gds.graph.project(
            'articleGraph',
            'Article',
            {
                SIMILAR: {
                    properties: 'similarity'
                }
            }
           )
    """
    )


def detect_community(tx):
    # logging.info("Detect community")
    tx.run(
        """
        CALL gds.louvain.write(
        'articleGraph',
        {
            writeProperty: 'community'
        }
        )
    """
    )

In [29]:
with GraphDatabase.driver(**NEO4J) as driver:
    with driver.session() as session:
        session.execute_write(clear_db)  # Clear the database
        for doc in documents:
            session.execute_write(create_graph_nodes, doc)
        session.execute_write(create_sim_edges, THRESHOLD)
        session.execute_write(create_graph_proj)
        session.execute_write(detect_community)

INFO:root:Clearing database
INFO:root:Create edges
INFO:neo4j.notifications:Received notification from DBMS server: {severity: INFORMATION} {code: Neo.ClientNotification.Statement.CartesianProduct} {category: PERFORMANCE} {title: This query builds a cartesian product between disconnected patterns.} {description: If a part of a query contains multiple disconnected patterns, this will build a cartesian product between all those parts. This may produce a large amount of data and slow down query processing. While occasionally intended, it may often be possible to reformulate the query that avoids the use of this cross product, perhaps by adding a relationship between the different parts or by using OPTIONAL MATCH (identifier is: (b))} {position: line: 2, column: 1, offset: 5} for query: '\n    MATCH (a:Article), (b:Article)\n    WHERE a.id < b.id\n    WITH a, b, gds.similarity.cosine(a.vector, b.vector) AS similarity\n    WHERE similarity > $threshold\n    CREATE (a)-[:SIMILAR {similarity:

### To drop graph projection 
If 'articleGraph' already exists

In [30]:
def drop_graph_projection(tx):
    result = tx.run(
        """
    CALL gds.graph.exists('articleGraph')
    YIELD exists
    RETURN exists
    """
    )
    if result.single()["exists"]:
        tx.run("CALL gds.graph.drop('articleGraph')")


with GraphDatabase.driver(**NEO4J) as driver:
    with driver.session() as session:
        session.execute_write(drop_graph_projection)

### To get difference between networkx and neo4j 

In [31]:
def return_community(tx):
    query = """
        MATCH (a:Article)
        RETURN a.community AS cluster, collect(a.title) AS articles
        ORDER BY cluster
        """
    result = tx.run(query)
    return [record for record in result]


with GraphDatabase.driver(**NEO4J) as driver:
    with driver.session() as session:
        records = session.execute_read(return_community)

neo4j_clusters = pd.DataFrame(
    records, columns=["Cluster Number", "Articles in Cluster"]
)
len(neo4j_clusters)

65

In [32]:
neo4j_clusters_grp_only = neo4j_clusters[
    neo4j_clusters["Articles in Cluster"].apply(lambda x: len(x) != 1)
]
len(neo4j_clusters_grp_only)

25

In [33]:
networkx = pd.read_csv("../artifacts/outputs/tfidf_cosine_graph_louvain_cluster.csv")
networkx_grp = networkx.groupby("cluster_label")["Page Title"].apply(set).reset_index()
networkx_grp_only = networkx_grp[
    networkx_grp["Page Title"].apply(lambda x: len(x) != 1)
]
len(networkx_grp_only)

26

In [34]:
first = neo4j_clusters_grp_only["Articles in Cluster"].to_list()
second = networkx_grp_only["Page Title"].to_list()


def calculate_overlap(set1, set2):
    return len(set(set1) & set(set2))


matches = []
matched_second_indices = set()

for f_index, f_set in enumerate(first):
    best_match = None
    best_overlap = -1
    best_match_index = -1
    for s_index, s_set in enumerate(second):
        overlap = calculate_overlap(f_set, s_set)
        if overlap > best_overlap:
            best_match = s_set
            best_overlap = overlap
            best_match_index = s_index
    matches.append((f_set, best_match))
    if best_match is not None:
        matched_second_indices.add(best_match_index)

# Add unmatched sets from first and second to the DataFrame
unmatched_first = [
    f_set for f_index, f_set in enumerate(first) if matches[f_index][1] is None
]
unmatched_second = [
    s_set
    for s_index, s_set in enumerate(second)
    if s_index not in matched_second_indices
]


matched_df = pd.DataFrame(matches, columns=["neo4j", "networkx"])

# Concatenate unmatched sets
for uf in unmatched_first:
    matched_df = pd.concat(
        [matched_df, pd.DataFrame([[uf, None]], columns=["neo4j", "networkx"])],
        ignore_index=True,
    )

for us in unmatched_second:
    matched_df = pd.concat(
        [matched_df, pd.DataFrame([[None, us]], columns=["neo4j", "networkx"])],
        ignore_index=True,
    )

In [35]:
matched_df["neo4j"] = matched_df["neo4j"].apply(
    lambda x: set(x) if str(x) != "None" else x
)
matched_df["Same/Different"] = matched_df.apply(
    lambda x: x["neo4j"] == x["networkx"], axis=1
)


def find_differences(set1, set2):
    set1 = set1 if set1 is not None else set()
    set2 = set2 if set2 is not None else set()
    if set1 == set2:
        return (
            None,
            None,
        )  # Return None or an empty string if there are no differences
    else:
        diff1 = set1.difference(set2)
        diff2 = set2.difference(set1)
        return (", ".join(diff1), ", ".join(diff2))


matched_df[["Only in neo4j", "Only in networkx"]] = matched_df.apply(
    lambda x: find_differences(x["neo4j"], x["networkx"]), axis=1, result_type="expand"
)

matched_df.to_excel("neo4j_networkx_cluster_comparison.xlsx", index=False)

### Query all nodes and edges from neo4j to be plotted in pyvis

In [36]:
query = """
MATCH (n)-[r]->(m)
RETURN n.title AS node_1, m.title AS node_2,
        r.similarity AS edge_weight, 
       n.ground_truth AS node_1_ground_truth, 
       m.ground_truth AS node_2_ground_truth, 
       n.community AS node_1_pred_cluster, 
       m.community AS node_2_pred_cluster,
       n.meta_desc AS node_1_meta,
       m.meta_desc AS node_2_meta
"""
# nodes with no relationship
query_2 = """MATCH (n)
WHERE NOT EXISTS ((n)--())
RETURN n.title AS node_title,
       n.ground_truth AS node_ground_truth,
       n.community AS node_community,
       n.meta_desc AS node_meta_desc
"""
with GraphDatabase.driver(**NEO4J) as driver:
    with driver.session() as session:
        results = session.run(query)
        data = pd.DataFrame(results.data())
        results_2 = session.run(query_2)
        data_2 = pd.DataFrame(results_2.data())

data["node_1"] = data["node_1"].astype(str)
data["node_2"] = data["node_2"].astype(str)
# data = data.dropna(subset=['node_2'])

data_2["node_community"] = ""


# Function to visualize the result
def visualize_result(df1, df2):
    visual_graph = pyvis.network.Network(select_menu=True, filter_menu=True)

    # Add nodes-nodes pair
    for _, row in df1.iterrows():
        # Add nodes
        visual_graph.add_node(
            row["node_1"],
            label=row["node_1"],
            title=f"Ground Truth: {row['node_1_ground_truth']}\nPredicted: {row['node_1_pred_cluster']}\nMeta Description: {row['node_1_meta']}",
            group=row["node_1_pred_cluster"],
        )
        visual_graph.add_node(
            row["node_2"],
            label=row["node_2"],
            title=f"Ground Truth: {row['node_2_ground_truth']}\nPredicted: {row['node_2_pred_cluster']}\nMeta Description: {row['node_2_meta']}",
            group=row["node_2_pred_cluster"],
        )

        # Add edge
        visual_graph.add_edge(
            row["node_1"], row["node_2"], title=f"Edge Weight: {row['edge_weight']}"
        )

    # Add solo nodes
    for _, row in df2.iterrows():
        visual_graph.add_node(
            row["node_title"],
            label=row["node_title"],
            title=f"Ground Truth: {row['node_ground_truth']}\nPredicted: No Community\nMeta Description: {row['node_meta_desc']}",
        )
    visual_graph.show("neo4j_network_tfidf_1.html", notebook=False)


visualize_result(data, data_2)

neo4j_network_tfidf_1.html
